$$\color{#8fd9fb}{ \texttt{ Arthur Lopes - Larissa Santana - Milene Almeida - Sabrina Pereira }}$$


$$\color{silver}{\small \texttt{Processamento de linguagem natural}}$$

$$\color{#6495ed}{\Large \texttt{TRABALHO PRÁTICO – IMPLEMENTAÇÃO DE SISTEMA RAG COM LANGCHAIN}}$$

$$\color{silver}{\small \texttt{Prof.  Vagner Macedo}}$$


---



## $\color{#6495ed}{\texttt{Tutor Para ENEM}}$

## Objetivo Geral

Aplicar técnicas de Mineração de Dados para identificar padrões de associação em transações comerciais, utilizando o algoritmo Apriori. Escolher com uma base de dados mais robusta, implementando funções personalizadas para cálculo de suporte, confiança e lift.

### Passo 1: Instalar as Bibliotecas Necessárias

In [15]:
!pip install -U langchain langchain-community langchain-google-genai pypdf faiss-cpu sentence-transformers streamlit pyngrok

### Passo 2: Ingestão de Dados (Scraping)

## Back-end

· Ingestão de dados - Leitura dos arquivos ou scraping de sites

- Scraping de provas do inep - https://download.inep.gov.br/enem/outros_documentos/enem_matriz_referencia.pdf

- Scraping dos E-books do Google Sites -  https://sites.google.com/educacao.quintana.sp.gov.br/biblioteca-virtual/livros-enem?pli=1&authuser=0



In [16]:
import os
import re
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import json

In [ ]:
# puxar materiais de estudo
url_livros = 'https://sites.google.com/educacao.quintana.sp.gov.br/biblioteca-virtual/livros-enem'
response = requests.get(url_livros)
soup = BeautifulSoup(response.text, 'html.parser')

# Pasta de destino
pasta_destino = 'livros_enem'
os.makedirs(pasta_destino, exist_ok=True)

file_ids = set()

texto_completo = response.text

# Procurando por links diretos limpos ou escapados no texto bruto
padrao_drive = re.compile(r'drive\.google\.com/(?:file/d/|open\?id=|uc\?id=|viewer\?.*?id=)([a-zA-Z0-9_-]{28,35})')
for match in padrao_drive.finditer(texto_completo):
    file_ids.add(match.group(1))

# Varredura específica em elementos de dados que o Google Sites moderno usa
for script in soup.find_all('script'):
    conteudo = script.string or ""
    if not conteudo:
        continue

    # Encontra IDs de recursos que começam por '1' seguidos de caracteres alfanuméricos e caracteres especiais permitidos
    for match in re.finditer(r'"(1[a-zA-Z0-9_-]{32})"', conteudo):
        file_ids.add(match.group(1))

    for match in re.finditer(r'\\"(1[a-zA-Z0-9_-]{32})\\"', conteudo):
        file_ids.add(match.group(1))

# Filtra IDs conhecidos de falsos positivos
file_ids = {fid for fid in file_ids if len(fid) == 33 and fid.startswith('1')}

print(f"\nTotal de arquivos únicos identificados: {len(file_ids)}")
if len(file_ids) > 0:
    print(f"Exemplo de IDs encontrados: {list(file_ids)[:5]}")

# Baixando os e-books convertendo o ID em link direto de download
baixados = 0
for idx, file_id in enumerate(sorted(file_ids)):
    download_url = f"https://drive.google.com/uc?id={file_id}&export=download"
    caminho_arquivo = os.path.join(pasta_destino, f"ebook_{file_id}.pdf")

    print(f"[{idx+1}/{len(file_ids)}] Tentando baixar arquivo id: {file_id}...")
    try:
        r = requests.get(download_url, stream=True, timeout=20)
        if r.status_code == 200:
            header_type = r.headers.get('Content-Type', '')
            if 'text/html' not in header_type:
                with open(caminho_arquivo, 'wb') as f:
                    for chunk in r.iter_content(chunk_size=8192):
                        f.write(chunk)
                print(f"   -> Sucesso! Salvo como {caminho_arquivo}")
                baixados += 1
            else:
                # Muitas vezes o Google Drive requer confirmação de vírus para arquivos grandes
                # Tentamos um bypass simples de confirmação de download do Google Drive
                confirm_session = requests.Session()
                res_confirm = confirm_session.get(download_url, timeout=20)
                token = None
                for k, v in res_confirm.cookies.items():
                    if k.startswith('download_warning'):
                        token = v
                if token:
                    confirm_url = f"https://drive.google.com/uc?id={file_id}&export=download&confirm={token}"
                    res_file = confirm_session.get(confirm_url, stream=True, timeout=20)
                    if 'text/html' not in res_file.headers.get('Content-Type', ''):
                        with open(caminho_arquivo, 'wb') as f:
                            for chunk in res_file.iter_content(chunk_size=8192):
                                f.write(chunk)
                        print(f"   -> Sucesso usando token de confirmação!")
                        baixados += 1
                        continue
                print(f"   -> Arquivo {file_id} requer login, está protegido ou é muito grande para download direto sem cookies.")
        else:
            print(f"   -> Falha ao baixar o arquivo {file_id} (Status: {r.status_code})")
    except Exception as e:
        print(f"   -> Erro ao baixar {file_id}: {e}")

print(f"\nProcesso concluído! Total de {baixados} arquivos baixados com sucesso na pasta: '{pasta_destino}'")

### Passo 3: Criação do Banco de Dados (Pipeline RAG)

In [ ]:
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

print("1. Ingestão: Lendo os e-books baixados...")
loader = PyPDFDirectoryLoader("livros_enem")
documentos = loader.load()

print("2. Chunking: Dividindo os textos para o LLM entender...")
# Divide em blocos de 1000 caracteres, sobrepondo 200 para não cortar raciocínios
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(documentos)

print("3. Vetorização e Armazenamento: Criando banco FAISS...")
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vector_store = FAISS.from_documents(chunks, embeddings)

# Salva o banco de dados processado localmente
vector_store.save_local("banco_faiss")
print("Banco de dados vetorial criado e salvo")

###Passo 4: Configuração da Chave da API

In [12]:
import os
from google.colab import userdata

# Puxa o valor da  chave de api
chave_secreta = userdata.get('tutorENEM')
os.environ["GOOGLE_API_KEY"] = chave_secreta

print("Chave de API configurada")

Chave de API configurada


## Front-end

O sistema deve permitir:

· Inserir perguntas em linguagem natural

· Recuperar informações relevantes da base

· Gerar respostas contextualizadas

· Apresentar respostas coerentes com a base utilizada





In [6]:
!pip install streamlit
!pip install pyngrok --quiet

In [22]:
%%writefile app.py
import streamlit as st

st.set_page_config(page_title="Tutor ENEM", page_icon="📚", layout="centered")
st.title("Tutor")
st.write("Olá, bem-vindo ao tutor para o enem!")

# ---------------------    SIDEBAR   ------------------------------
with st.sidebar:
    st.header("Menu")
    if st.button("Limpar Conversa", use_container_width=True):
        st.session_state.messages = []
        st.rerun()

st.sidebar.button("Histórico")

# ---------------------     MAIN     ------------------------------
if "messages" not in st.session_state:
    st.session_state.messages = []

# Mensagens anteriores
for message in st.session_state.messages:
    with st.chat_message(message["role"]):
        st.markdown(message["content"])
#nova pergunta
if prompt := st.chat_input("Digite sua dúvida sobre o ENEM..."):

    st.session_state.messages.append({"role": "user", "content": prompt})
    with st.chat_message("user"):
        st.markdown(prompt)


Overwriting app.py


### Parte 6: O Back-end (Injetando o RAG)

In [23]:
%pip install -q langchain-classic

In [28]:
%%writefile -a app.py

import os
import streamlit as st
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

#streamlit pega a chave do ambiente do Colab
api_key = os.environ.get("GOOGLE_API_KEY")

#função que carrega o banco salvo no Passo 2
@st.cache_resource
def carregar_banco():
    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
    banco = FAISS.load_local("banco_faiss", embeddings, allow_dangerous_deserialization=True)
    return banco.as_retriever(search_kwargs={"k": 4})

retriever = carregar_banco()

#verifica o session_state já criado.
if st.session_state.messages and st.session_state.messages[-1]["role"] == "user":

    pergunta_usuario = st.session_state.messages[-1]["content"]

    with st.chat_message("assistant"):
        with st.spinner("Buscando informações nos e-books..."):

            #geração de Resposta passando a api_key
            llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite", temperature=0.3, google_api_key=api_key)

            system_prompt = (
                "Você é um tutor especializado na preparação para o ENEM. "
                "Responda diretamente à dúvida do estudante, com linguagem "
                "didática e foco no exame. "
                "Use os trechos recuperados como material de apoio somente "
                "quando forem relevantes à pergunta. "
                "Se os trechos forem insuficientes ou tratarem de outro assunto, "
                "use seu conhecimento geral e informe brevemente que a "
                "orientação não foi encontrada nos materiais consultados. "
                "Não force relações com trechos irrelevantes. "
                "Não invente datas, editais ou informações atuais do exame; "
                "nesses casos, oriente consultar o Inep. "
                "Ao sugerir estudos, apresente assuntos e exemplos práticos. "
                "\n\nMateriais recuperados:\n{context}"
            )
            prompt_template = ChatPromptTemplate.from_messages([
                ("system", system_prompt),
                ("human", "{input}"),
            ])

            #conecta o mecanismo de busca com o LLM
            chain_pergunta = create_stuff_documents_chain(llm, prompt_template)
            rag_chain = create_retrieval_chain(retriever, chain_pergunta)

            resposta = rag_chain.invoke({"input": pergunta_usuario})
            texto_resposta = resposta["answer"]

            #exibe a resposta e salva no histórico do front
            st.markdown(texto_resposta)
            st.session_state.messages.append({"role": "assistant", "content": texto_resposta})

Appending to app.py


In [33]:
from google.colab import userdata
from pyngrok import conf

conf.get_default().auth_token = userdata.get("NGROK_AUTHTOKEN")

In [34]:
from pyngrok import ngrok
import os

ngrok.kill()

public_url = ngrok.connect(8501)
print(f"Acesse o seu app Streamlit aqui: {public_url}")

# Executa o Streamlit
!streamlit run app.py --server.port 8501

Acesse o seu app Streamlit aqui: NgrokTunnel: "https://thesis-congrats-excusable.ngrok-free.dev" -> "http://localhost:8501"


2026-11-10 04:51:57.699 Uvicorn server started on :::8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://136.85.197.90:8501

/content/app.py:34: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.embeddings import HuggingFaceEmbeddings
/content/app.py:46: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as